# WindShift — Gemini cross-model reviewer validation

**Independent follow-up, not a preregistered replication.** Holds GPT-originating peer reports and current simulated observations fixed, and substitutes a Gemini reviewing agent. Thus it tests reviewer-model generality under identical prompts, not a complete Gemini-only multi-agent swarm. **Calls are disabled by default.** Model: `google/gemini-3.1-flash-lite`.

No private key is embedded. 8-call feasibility pilot, then up to 192 reviewer calls total; resume uses previous records. No unsupported scientific conclusions are printed before all calls complete.

In [ ]:
README_TEXT = "# Gemini reviewer-model cross-family replication\n\nThis study takes the 192 EXACT reviewer prompts from WindShift v3+v4 (24 episodes x 2 post windows x four factorial arms) and evaluates a Google Gemini-family model under unchanged system prompts, current evidence, and GPT-generated peer reports.\n\n**Scope:** tests cross-model robustness of the *reviewing agent's response to identical advice*. This does NOT reproduce the full Gemini-to-Gemini swarm; the peer reports are intentionally held fixed from the GPT study, so any new difference is attributable to the reviewer-model change conditional on those inputs. The old- or new-timestamp manipulations remain synthetic counterfactual metadata.\n\nCore metric: correct decisions after source reversals, with separate wrong-source and abstention counts; ambiguous-case unsupported choices; factorial content/timestamp contrasts. One sample per prompt and one alternative model cannot establish universal robustness.\n\nNo API key is stored. Model calls disabled by default. Pilot: 8 calls; full: up to 192 success records including pilot, with checkpointing. Failed provider responses are logged, and you should inspect before retrying: providers may charge for failures.\n\nDataset manifest SHA256: 860d2d9cd4b3690bf6c0b2ad701bb773d04aa4633f4c914647d3b36a777e3417.\n"
EMBEDDED = (
    '...'
    'VApRKUSlEBL8PAn+z/8flcuPpQ=='
)
EXPECTED_MANIFEST_SHA = '860d2d9cd4b3690bf6c0b2ad701bb773d04aa4633f4c914647d3b36a777e3417'
# Setup: source data are embedded; no ZIP upload needed.
import base64,zlib,json,hashlib
MANIFEST=json.loads(zlib.decompress(base64.b64decode(EMBEDDED)).decode('utf-8'))
MANIFEST_SHA=hashlib.sha256(json.dumps(MANIFEST,ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()).hexdigest()
assert MANIFEST_SHA==EXPECTED_MANIFEST_SHA
print('Loaded 192 frozen, verified reviewer prompts. SHA256:',MANIFEST_SHA)


In [ ]:
import os, json, hashlib, urllib.request, urllib.error, time, random, statistics
from pathlib import Path

MODEL='google/gemini-3.1-flash-lite'
TEMPERATURE=0.2
MAX_TOKENS=1024
OUT=Path('/content/windshift_gemini_replication_calls.jsonl')
VALID={'WEST','EAST','ABSTAIN'}
ARMS=('fresh_peer','current_content_old_timestamp','historical_content_new_timestamp','stale_peer')

def sha(s):
    return hashlib.sha256(s.encode('utf-8')).hexdigest()

def parse_reply(raw):
    s=raw.strip()
    if s.startswith('```'):
        s='\n'.join(s.splitlines()[1:-1]).strip()
    try: d=json.loads(s)
    except json.JSONDecodeError:
        i=s.find('{'); j=s.rfind('}')
        if i<0 or j<i: raise ValueError('No valid JSON object')
        d=json.loads(s[i:j+1])
    if d.get('source') not in VALID: raise ValueError('Invalid choice '+str(d.get('source')))
    c=float(d.get('confidence',0.5))
    if c<0 or c>1: raise ValueError('Invalid confidence')
    return {'source':d['source'],'confidence':c,'reason':str(d.get('reason',''))[:1000]}

def rows():
    if not OUT.exists():return []
    ans=[]
    for n,line in enumerate(OUT.read_text().splitlines(),1):
        if line.strip():
            try:ans.append(json.loads(line))
            except Exception as e:raise ValueError(f'Corrupted output line {n}: {e}')
    return ans

def audit_manifest():
    assert len(MANIFEST)==192
    ids=[(r['episode'],r['hour'],r['arm']) for r in MANIFEST]
    assert len(set(ids))==192
    for r in MANIFEST:
        assert sha(r['full_prompt'])==r['prompt_sha256']
        assert 'truth' not in r['full_prompt'] and 'oracle_contrib' not in r['full_prompt']
        assert r['truth'] in VALID
        assert json.loads(r['full_prompt'])['observations']['now_hour']==r['hour']
    for eid,h in {(r['episode'],r['hour']) for r in MANIFEST}:
        rs=[r for r in MANIFEST if r['episode']==eid and r['hour']==h]
        assert len(rs)==4 and {r['arm'] for r in rs}==set(ARMS)
        assert len({json.dumps(json.loads(r['full_prompt'])['observations'],sort_keys=True) for r in rs})==1
        assert len({r['system_prompt'] for r in rs})==1
    print('PREFLIGHT PASS: 192 exact frozen reviewer prompts; four matched arms; no oracle leakage')

def call_provider(key,r):
    body={'model':MODEL,'temperature':TEMPERATURE,'max_tokens':MAX_TOKENS,
          'messages':[{'role':'system','content':r['system_prompt']},
                      {'role':'user','content':r['full_prompt']}]}
    request=urllib.request.Request('https://openrouter.ai/api/v1/chat/completions',
            data=json.dumps(body).encode(),method='POST',headers={
            'Authorization':'Bearer '+key,'Content-Type':'application/json','X-Title':'WindShift Gemini Replication'})
    with urllib.request.urlopen(request,timeout=120) as resp:
        response=json.loads(resp.read().decode())
    if not response.get('choices'): raise RuntimeError('Missing model response '+str(response)[:300])
    raw=response['choices'][0]['message'].get('content')
    if not isinstance(raw,str) or not raw.strip():
        raise ValueError('Empty or non-text model response')
    return raw,response.get('id'),response.get('usage',{}),response.get('model',MODEL)

def run(max_new=8,authorize=False):
    audit_manifest()
    prev=rows(); done={}
    for r in prev:
        if r.get('status')=='ok':
            if r['manifest_sha256']!=MANIFEST_SHA or r['model_requested']!=MODEL:
                raise ValueError('Cannot resume with a different manifest/model')
            k=(r['episode'],r['hour'],r['arm'])
            if k in done:raise ValueError('Duplicated successful task '+str(k))
            done[k]=r
    pending=[r for r in MANIFEST if (r['episode'],r['hour'],r['arm']) not in done]
    print(f'MODEL {MODEL}; {len(done)}/192 already complete; {len(pending)} pending; requested new success cap {max_new}')
    if not authorize or max_new<=0:
        print('DRY RUN. No inference calls. Check model pricing and explicitly authorize first.')
        return
    from google.colab import userdata
    key=userdata.get('OPENROUTER_API_KEY')
    if not key:raise RuntimeError('Add OPENROUTER_API_KEY to Colab Secrets')
    num=0
    with OUT.open('a') as f:
        for src in pending:
            if num>=max_new:break
            rec={k:src[k] for k in ('episode','kind','hour','truth','arm','prompt_sha256','original_gpt_choice')}
            rec.update({'manifest_sha256':MANIFEST_SHA,'model_requested':MODEL,
                        'temperature':TEMPERATURE,'max_tokens':MAX_TOKENS,
                        'status':'attempt','time_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())})
            try:
                raw,pid,usage,actual=call_provider(key,src)
                rec.update({'raw_response':raw,'provider_request_id':pid,'usage':usage,
                            'model_returned':actual,'choice':parse_reply(raw),'status':'ok'})
                if not pid: raise RuntimeError('Provider request ID missing')
                f.write(json.dumps(rec,ensure_ascii=False)+'\n'); f.flush()
                num+=1
                print(f"{len(done)+num}/192 {src['episode']} {src['hour']} {src['arm']} -> {rec['choice']['source']}")
            except Exception as e:
                rec.update({'status':'error','error':str(e)[:500]})
                f.write(json.dumps(rec,ensure_ascii=False)+'\n'); f.flush()
                print('STOPPED after logged error:',str(e)[:400]);break
    print('New successful:',num,'; total:',len(done)+num)

def summarize():
    recent=rows(); completed={}
    for r in recent:
        if r.get('status')!='ok':continue
        assert r['manifest_sha256']==MANIFEST_SHA and r['model_requested']==MODEL
        k=(r['episode'],r['hour'],r['arm'])
        if k in completed:raise ValueError('Duplicate success '+str(k))
        assert r['choice']==parse_reply(r['raw_response'])
        completed[k]=r
    print('GEMINI COMPLETED',len(completed),'/192; errors',sum(r['status']=='error' for r in recent))
    if not completed:return
    from collections import Counter
    for kind in ('switch','stable','ambiguous'):
        for arm in ARMS:
            items=[r for r in completed.values() if r['kind']==kind and r['arm']==arm]
            if not items:continue
            counts=Counter(r['choice']['source'] for r in items)
            correct=sum(r['choice']['source']==r['truth'] for r in items)
            wrong=sum(r['choice']['source'] not in ('ABSTAIN',r['truth']) for r in items)
            abstain=counts.get('ABSTAIN',0)
            print(kind,arm,'n',len(items),'correct',correct,'abstain',abstain,'unsupported',wrong)
    if len(completed)!=192:
        print('PARTIAL DATA: no final comparison reported. Export logs for auditing.')
        return
    effects=[]
    for ep in sorted(set(r['episode'] for r in completed.values() if r['kind']=='switch')):
        def avg(arm):
            a=[r for r in completed.values() if r['episode']==ep and r['arm']==arm]
            assert len(a)==2
            return sum(r['choice']['source']==r['truth'] for r in a)/2
        effects.append((avg('fresh_peer')-avg('stale_peer'),
                        avg('current_content_old_timestamp')-avg('historical_content_new_timestamp')))
    print('Complete 16-switch-episode descriptive paired contrasts:',effects)

# Validate manifest and ensure no accidental API calls at notebook startup.
audit_manifest()


## Pilot (eight provider requests)
Check OpenRouter model availability, pricing and the available balance. Add `OPENROUTER_API_KEY` in Colab Secrets, then deliberately switch `AUTHORIZE_PILOT` to `True`. Response failures may still incur charges and cause execution to stop.

In [ ]:
AUTHORIZE_PILOT=False
run(max_new=8,authorize=AUTHORIZE_PILOT)
summarize()


## Full reviewer validation (up to 192 total)
After the pilot succeeds and you have checked the returned decisions and charges, authorize. The runner resumes completed calls, requesting at most 192 additional successes and stopping when all 192 are completed.

In [ ]:
AUTHORIZE_FULL=False
run(max_new=192,authorize=AUTHORIZE_FULL)
summarize()


## Export auditable evidence ZIP
Use the export cell after the pilot or the complete experiment. The ZIP will contain the frozen prompt manifest, new Gemini calls, README and the original-manifest fingerprint. Upload the result to ChatGPT for reanalysis.

In [ ]:
import zipfile
from google.colab import files
archive='/content/WindShift_Gemini_Reviewer_Results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as zz:
    zz.writestr('README.md',README_TEXT)
    zz.writestr('frozen_reviewer_prompts.json',json.dumps(MANIFEST,ensure_ascii=False,indent=2))
    if OUT.exists(): zz.write(OUT,'gemini_reviewer_calls.jsonl')
    zz.writestr('manifest_sha256.txt',MANIFEST_SHA+'\n')
print('Exporting',archive)
files.download(archive)
